# **_Init_**

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length




# **_Reading the table from bronze_**

In [0]:
df = spark.table("bronze.crm_sales_details")
df.show()


# _**Transformation**_
 _** Trimming 
**_

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name,trim(col(field.name)))
df.show()


# **Cleaning the dates columns**

In [0]:
df.select("sls_order_dt").printSchema()

In [0]:
from pyspark.sql import functions as F

def clean_yyyymmdd_date(col_name: str):
    c = F.col(col_name).cast('string')
    return (
        F.when((c == 0) | (F.length(c) != 8), F.lit(None).cast("date"))
         .otherwise(F.to_date(c.cast("string"), "yyyyMMdd"))
    )

df = (
    df
    .withColumn("sls_order_dt", clean_yyyymmdd_date("sls_order_dt"))
    .withColumn("sls_ship_dt",  clean_yyyymmdd_date("sls_ship_dt"))
    .withColumn("sls_due_dt",   clean_yyyymmdd_date("sls_due_dt"))
)
# def clean_yyyymmdd_date(col_name:str):
#   c = F.col(col_name)

#   already_date = c.cast('date')

#   c_str = c.cast('string')
#   parsed_yyyymmdd = F.to_date(c_str,'yyyymmdd')

#   return F.when(already_date.isNotNull(),already_date)\
#           .when((c_str.isNull()) | (c_str == '0')| (F.length(c_str) != 8), F.lit(None).cast('date'))




# df = (
#     df
#     .withColumn(
#         "sls_order_dt",
#         F.when(
#             (col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8),
#             None
#         ).otherwise(F.to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
#     )
#     .withColumn(
#         "sls_ship_dt",
#         F.when(
#             (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8),
#             None
#         ).otherwise(F.to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
#     )
#     .withColumn(
#         "sls_due_dt",
#         F.when(
#             (col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8),
#             None
#         ).otherwise(F.to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
#     )
# )
# df.show()

df.show()


# **_Sales and price Column Calulations_**

In [0]:
df = (
    df
    .withColumn(
        'sls_price',
        F.when(
            (col('sls_price').isNull()) | (col('sls_price') <= 0),
            F.when(
                col('sls_quantity') != 0,
                col('sls_sales') / col('sls_quantity')
            ).otherwise(None)

        ).otherwise(col('sls_price'))
    )
)


# **_Renaming the Columns_**

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)
df.show()

# **_Writing into the Silver table_**

In [0]:
spark.sql("DROP TABLE IF EXISTS silver.crm_sales_details")

df.write.mode("overwrite").format("delta").saveAsTable("silver.crm_sales_details")

In [0]:
%sql
select * from silver.crm_sales_details limit 20;